In [1]:
### Extract file in /Users/wenjun/Downloads/JPXData/AGM

import os
import zipfile
import shutil
from pathlib import Path

# 设置源目录和目标目录
source_dir = "/Users/wenjun/Downloads/JPXData/AGM"
target_dir = "/Users/wenjun/Downloads/JPXData/AGM/XBRL"

# 确保目标目录存在
os.makedirs(target_dir, exist_ok=True)

def extract_xbrl_from_zip(zip_path, target_dir):
    """从ZIP文件中提取XBRL文件"""
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # 获取ZIP文件中的所有文件列表
            file_list = zip_ref.namelist()
            
            # 查找XBRL文件
            xbrl_files = [f for f in file_list if f.endswith('.xbrl')]
            
            if not xbrl_files:
                print(f"在 {zip_path} 中没有找到XBRL文件")
                return
            
            # 获取zip文件的基本名称（不含扩展名）
            zip_base_name = os.path.splitext(os.path.basename(zip_path))[0]
            
            # 构建新的目标文件名
            target_path = os.path.join(target_dir, f"{zip_base_name}.xbrl")
            
            # 提取第一个XBRL文件并重命名
            with zip_ref.open(xbrl_files[0]) as source, open(target_path, 'wb') as target:
                shutil.copyfileobj(source, target)
            print(f"已提取并重命名为: {zip_base_name}.xbrl")
                
    except Exception as e:
        print(f"处理 {zip_path} 时出错: {str(e)}")

def main():
    # 获取所有ZIP文件
    zip_files = [f for f in os.listdir(source_dir) if f.endswith('.zip')]
    
    if not zip_files:
        print(f"在 {source_dir} 中没有找到ZIP文件")
        return
    
    print(f"找到 {len(zip_files)} 个ZIP文件")
    
    # 处理每个ZIP文件
    for zip_file in zip_files:
        zip_path = os.path.join(source_dir, zip_file)
        print(f"\n处理: {zip_file}")
        extract_xbrl_from_zip(zip_path, target_dir)

if __name__ == "__main__":
    main() 

找到 505 个ZIP文件

处理: 41920_20250327.zip
已提取并重命名为: 41920_20250327.xbrl

处理: 25020_20250327.zip
已提取并重命名为: 25020_20250327.xbrl

处理: 89140_20250327.zip
已提取并重命名为: 89140_20250327.xbrl

处理: 66350_20250328.zip
已提取并重命名为: 66350_20250328.xbrl

处理: 34750_20250131.zip
已提取并重命名为: 34750_20250131.xbrl

处理: 291A0_20250327.zip
已提取并重命名为: 291A0_20250327.xbrl

处理: 56180_20250327.zip
已提取并重命名为: 56180_20250327.xbrl

处理: 60490_20250131.zip
已提取并重命名为: 60490_20250131.xbrl

处理: 99720_20250228.zip
已提取并重命名为: 99720_20250228.xbrl

处理: 33460_20250225.zip
已提取并重命名为: 33460_20250225.xbrl

处理: 228A0_20250228.zip
已提取并重命名为: 228A0_20250228.xbrl

处理: 39640_20250326.zip
已提取并重命名为: 39640_20250326.xbrl

处理: 70440_20250327.zip
已提取并重命名为: 70440_20250327.xbrl

处理: 25870_20250328.zip
已提取并重命名为: 25870_20250328.xbrl

处理: 34800_20250129.zip
已提取并重命名为: 34800_20250129.xbrl

处理: 70370_20250321.zip
已提取并重命名为: 70370_20250321.xbrl

处理: 59460_20250327.zip
已提取并重命名为: 59460_20250327.xbrl

处理: 43770_20250327.zip
已提取并重命名为: 43770_20250327.xbrl

处理: 92790_202

In [2]:
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import html
import pandas as pd
import logging
from datetime import datetime
import os

# 设置路径
xbrl_dir = "/Users/wenjun/Downloads/JPXData/AGM/XBRL"
tables_dir = "/Users/wenjun/Downloads/JPXData/AGM/XBRL/Tables"
log_dir = "/Users/wenjun/Downloads/JPXData/AGM/Logs"

# 确保输出目录和日志目录存在
os.makedirs(tables_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# 设置日志
log_file = os.path.join(log_dir, f"agm_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)

def find_resolution_elements(root, namespaces):
    """查找所有可能的决议相关元素"""
    resolution_keywords = [
        'resolution', 'meeting', 'shareholder', '決議', '議決', '株主', '総会',
        '議案', '議事', '採決', '投票', '賛成', '反対', '可決', '否決'
    ]
    
    found_elements = []
    for elem in root.iter():
        if '}' in elem.tag:
            element_name = elem.tag.split('}')[1].lower()
            if any(keyword in element_name for keyword in resolution_keywords):
                found_elements.append(elem.tag)
                logging.debug(f"找到决议相关元素: {element_name}")
    
    return found_elements

def html_table_to_dataframe(table):
    """将HTML表格转换为DataFrame"""
    rows = []
    for tr in table.find_all('tr'):
        row = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
        if row:  # 只添加非空行
            rows.append(row)
    
    if not rows:
        return None
    
    # 使用第一行作为列名
    headers = rows[0]
    data = rows[1:]
    
    # 确保所有行的列数一致
    max_cols = max(len(row) for row in data)
    if len(headers) < max_cols:
        # 如果表头列数不足，添加额外的列名
        headers.extend([f'Column_{i+1}' for i in range(len(headers), max_cols)])
    
    # 确保所有数据行的列数一致
    for row in data:
        if len(row) < max_cols:
            row.extend([''] * (max_cols - len(row)))
    
    df = pd.DataFrame(data, columns=headers)
    return df

def process_xbrl_file(file_path):
    """处理单个XBRL文件并提取表格"""
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        
        logging.info(f"处理文件: {base_name}")
        
        resolution_elements = find_resolution_elements(root, {})
        if not resolution_elements:
            logging.warning(f"在文件 {file_path} 中未找到决议相关元素")
            return None
        
        logging.info(f"找到 {len(resolution_elements)} 个可能的决议相关元素")
        
        for elem_path in resolution_elements:
            elements = root.findall(f'.//{elem_path}')
            for i, element in enumerate(elements, 1):
                if element.text and element.text.strip():
                    text = html.unescape(element.text)
                    logging.debug(f"处理元素内容: {text[:200]}...")
                    
                    # 尝试解析HTML
                    soup = BeautifulSoup(text, 'html.parser')
                    tables = soup.find_all('table')
                    
                    if tables:
                        for j, table in enumerate(tables, 1):
                            df = html_table_to_dataframe(table)
                            if df is not None:
                                # 保存原始表格
                                output_file = os.path.join(tables_dir, f"{base_name}.csv")
                                df.to_csv(output_file, index=False, encoding='utf-8-sig')
                                logging.debug(f"已保存表格到: {output_file}")
        
        return True
    
    except Exception as e:
        logging.error(f"处理文件 {file_path} 时出错: {str(e)}", exc_info=True)
        return None

def main(n=None):
    """
    主程序
    Args:
        n (int, optional): 要处理的文件数量，None表示处理所有文件
    """
    xbrl_files = [f for f in os.listdir(xbrl_dir) if f.endswith('.xbrl')]
    
    # 如果指定了n，则只处理前n个文件
    if n is not None:
        n = min(n, len(xbrl_files))  # 确保n不超过文件总数
        xbrl_files = xbrl_files[:n]
        logging.info(f"将处理前 {n} 个XBRL文件（共 {len(xbrl_files)} 个文件）")
    else:
        logging.info(f"将处理所有 {len(xbrl_files)} 个XBRL文件")

    for i, file_name in enumerate(xbrl_files, 1):
        logging.info(f"处理第 {i}/{len(xbrl_files)} 个文件: {file_name}")
        file_path = os.path.join(xbrl_dir, file_name)
        
        # 处理文件内容
        process_xbrl_file(file_path)

if __name__ == "__main__":
    # 可以在这里指定要处理的文件数量
    
    #n = 5  # 比如处理前5个文件，设置为None则处理所有文件
    main()

2025-04-01 00:14:57,994 - INFO - 将处理所有 505 个XBRL文件
2025-04-01 00:14:57,995 - INFO - 处理第 1/505 个文件: 74220_20250318.xbrl
2025-04-01 00:14:57,998 - INFO - 处理文件: 74220_20250318
2025-04-01 00:14:57,999 - INFO - 找到 1 个可能的决议相关元素
2025-04-01 00:14:58,016 - INFO - 处理第 2/505 个文件: 48830_20250328.xbrl
2025-04-01 00:14:58,017 - INFO - 处理文件: 48830_20250328
2025-04-01 00:14:58,018 - INFO - 找到 1 个可能的决议相关元素
2025-04-01 00:14:58,030 - INFO - 处理第 3/505 个文件: 48960_20250328.xbrl
2025-04-01 00:14:58,031 - INFO - 处理文件: 48960_20250328
2025-04-01 00:14:58,032 - INFO - 找到 1 个可能的决议相关元素
2025-04-01 00:14:58,035 - INFO - 处理第 4/505 个文件: 43940_20250303.xbrl
2025-04-01 00:14:58,037 - INFO - 处理文件: 43940_20250303
2025-04-01 00:14:58,037 - INFO - 找到 1 个可能的决议相关元素
2025-04-01 00:14:58,045 - INFO - 处理第 5/505 个文件: 36230_20250326.xbrl
2025-04-01 00:14:58,047 - INFO - 处理文件: 36230_20250326
2025-04-01 00:14:58,047 - INFO - 找到 1 个可能的决议相关元素
2025-04-01 00:14:58,055 - INFO - 处理第 6/505 个文件: 78270_20250131.xbrl
2025-04-01 00:14:58,057 - 

In [49]:
import os
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import html
import re
import pandas as pd
from collections import defaultdict
from datetime import datetime
import logging

# 在notebook开头添加
import logging

# 配置日志只输出到终端
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 测试日志是否正常工作
logging.info("日志系统初始化完成")

def clean_table_data(df):
    """清理表格数据"""
    # 打印列名，用于调试
    logging.info(f"原始列名: {df.columns.tolist()}")
    
    # 处理可决要件和结果混在一起的情况
    result_columns = [col for col in df.columns if '決議' in col or '可決' in col]
    if result_columns:
        # 创建新的列
        df['決議の結果'] = ''
        df['賛成割合（％）'] = ''
        
        # 分离结果和百分比
        for idx, row in df.iterrows():
            result_text = str(row[result_columns[0]])
            
            # 处理可决/否决结果
            if '可決' in result_text:
                df.at[idx, '決議の結果'] = '可決'
            elif '否決' in result_text:
                df.at[idx, '決議の結果'] = '否決'
            
            # 提取百分比 - 使用更灵活的正则表达式
            percent_match = re.search(r'(\d+\.?\d*)%', result_text)
            if percent_match:
                df.at[idx, '賛成割合（％）'] = percent_match.group(1)
            else:
                # 尝试其他格式的百分比
                alt_match = re.search(r'(\d+\.?\d*)％', result_text)
                if alt_match:
                    df.at[idx, '賛成割合（％）'] = alt_match.group(1)
                else:
                    # 尝试从賛成数（個）和反対数（個）计算
                    agree_cols = [col for col in df.columns if '賛成' in col]
                    oppose_cols = [col for col in df.columns if '反対' in col]
                    if agree_cols and oppose_cols:
                        try:
                            agree = float(str(row[agree_cols[0]]).replace(',', ''))
                            oppose = float(str(row[oppose_cols[0]]).replace(',', ''))
                            if agree + oppose > 0:
                                rate = (agree / (agree + oppose)) * 100
                                df.at[idx, '賛成割合（％）'] = f"{rate:.2f}"
                        except:
                            pass
    
    # 处理数据挤在一起的情况
    for col in df.columns:
        if df[col].dtype == 'object':
            # 尝试分割挤在一起的数据
            df[col] = df[col].apply(lambda x: x.split('19,')[0] if isinstance(x, str) and '19,' in x else x)
    
    # 删除空的Column_7列
    if 'Column_7' in df.columns and df['Column_7'].isna().all():
        df = df.drop('Column_7', axis=1)
    
    # 重新排列列顺序
    desired_columns = [
        '決議事項', '賛成数（個）', '反対数（個）', '棄権数（個）', 
        '可決要件', '決議の結果', '賛成割合（％）'
    ]
    
    # 只保留存在的列
    existing_columns = [col for col in desired_columns if col in df.columns]
    df = df[existing_columns]
    
    return df

def html_table_to_dataframe(table):
    """将HTML表格转换为DataFrame"""
    rows = []
    for tr in table.find_all('tr'):
        row = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
        if row:  # 只添加非空行
            rows.append(row)
    
    if not rows:
        return None
    
    # 使用第一行作为列名
    headers = rows[0]
    data = rows[1:]
    
    # 确保所有行的列数一致
    max_cols = max(len(row) for row in data)
    if len(headers) < max_cols:
        # 如果表头列数不足，添加额外的列名
        headers.extend([f'Column_{i+1}' for i in range(len(headers), max_cols)])
    
    # 确保所有数据行的列数一致
    for row in data:
        if len(row) < max_cols:
            row.extend([''] * (max_cols - len(row)))
    
    df = pd.DataFrame(data, columns=headers)
    
    # 清理数据
    df = clean_table_data(df)
    
    return df

def extract_approval_rate(df):
    """提取最低赞成率"""
    if '賛成割合（％）' not in df.columns:
        logging.info("未找到賛成割合（％）列")
        return None
    
    rates = []
    for idx, rate in df['賛成割合（％）'].items():
        if pd.notna(rate) and rate != '':
            try:
                # 移除百分号并转换为浮点数
                rate_value = float(str(rate).replace('%', '').replace('％', ''))
                rates.append(rate_value)
                logging.info(f"找到赞成率: {rate_value}% (行 {idx+1})")
            except Exception as e:
                logging.warning(f"无法解析赞成率 '{rate}' (行 {idx+1}): {str(e)}")
                continue
    
    if rates:
        min_rate = min(rates)
        logging.info(f"最低赞成率: {min_rate}%")
        return min_rate
    else:
        logging.info("未找到有效的赞成率")
        return None

def process_xbrl_file(file_path):
    """处理单个XBRL文件"""
    try:
        # 解析XML文件
        tree = ET.parse(file_path)
        root = tree.getroot()
        
        # 获取文件名（不含扩展名）用于输出文件
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        
        # 尝试获取公司名称
        company_name = None
        for elem in root.iter():
            if '}' in elem.tag:
                element_name = elem.tag.split('}')[1].lower()
                if 'company' in element_name or 'name' in element_name or '会社' in element_name:
                    if elem.text and elem.text.strip():
                        company_name = elem.text.strip()
                        logging.info(f"找到公司名称: {company_name}")
                        break
        
        # 查找所有可能的决议相关元素
        resolution_elements = find_resolution_elements(root, {})
        
        if not resolution_elements:
            logging.info(f"文件 {base_name} 中未找到决议相关元素")
            return None
        
        logging.info(f"处理文件: {base_name} (公司: {company_name if company_name else '未知'})")
        
        # 处理每个找到的元素
        for elem_path in resolution_elements:
            elements = root.findall(f'.//{elem_path}')
            logging.info(f"找到 {len(elements)} 个决议元素")
            
            for i, element in enumerate(elements, 1):
                if element.text and element.text.strip():
                    text = html.unescape(element.text)
                    
                    # 尝试解析HTML
                    soup = BeautifulSoup(text, 'html.parser')
                    tables = soup.find_all('table')
                    
                    if tables:
                        logging.info(f"在决议元素 {i} 中找到 {len(tables)} 个表格")
                        
                        for j, table in enumerate(tables, 1):
                            # 转换为DataFrame
                            df = html_table_to_dataframe(table)
                            if df is not None:
                                logging.info(f"处理表格 {j} 的列: {df.columns.tolist()}")
                                
                                # 提取最低赞成率
                                min_rate = extract_approval_rate(df)
                                
                                # 检查是否有否决的决议
                                rejection_flag = 'N'
                                if '決議の結果' in df.columns:
                                    rejection_flag = 'Y' if df['決議の結果'].str.contains('否決', na=False).any() else 'N'
                                    logging.info(f"决议结果: {'有否决' if rejection_flag == 'Y' else '全部通过'}")
                                else:
                                    # 尝试在其他列中查找否决信息
                                    for col in df.columns:
                                        if df[col].astype(str).str.contains('否決', na=False).any():
                                            rejection_flag = 'Y'
                                            logging.info(f"在列 {col} 中发现否决")
                                            break
                                
                                return {
                                    'file_name': f"{base_name}",
                                    'company_name': company_name if company_name else 'N/A',
                                    'min_approval_rate': min_rate if min_rate is not None else 'N/A',
                                    'rejection_flag': rejection_flag
                                }
    
    except Exception as e:
        logging.error(f"处理文件 {file_path} 时出错: {str(e)}", exc_info=True)
    
    return None

def find_resolution_elements(root, namespaces):
    """查找所有可能的决议相关元素"""
    resolution_keywords = [
        'resolution', 'meeting', 'shareholder', '決議', '議決', '株主', '総会'
    ]
    
    found_elements = []
    for elem in root.iter():
        if '}' in elem.tag:
            element_name = elem.tag.split('}')[1].lower()
            if any(keyword in element_name for keyword in resolution_keywords):
                found_elements.append(elem.tag)
    
    return found_elements

def process_all_files(n=None):
    """处理所有文件并生成汇总报告"""
    # 设置路径
    xbrl_dir = "/Users/wenjun/Downloads/JPXData/AGM/XBRL"
    output_dir = "/Users/wenjun/Downloads/Quants/EDINET"
    
    # 确保输出目录存在
    os.makedirs(output_dir, exist_ok=True)
    
    results = []
    xbrl_files = [f for f in os.listdir(xbrl_dir) if f.endswith('.xbrl')]
    
    #print(len(xbrl_files))
    # 如果指定了n，则只处理前n个文件
    if n is not None:
        n = min(n, len(xbrl_files))
        xbrl_files = xbrl_files[:n]
        logging.info(f"将处理前 {n} 个文件（共有 {len(xbrl_files)} 个XBRL文件）")
    else:
        logging.info(f"将处理所有 {len(xbrl_files)} 个XBRL文件")
    
    for i, file_name in enumerate(xbrl_files, 1):
        file_path = os.path.join(xbrl_dir, file_name)
        logging.info(f"[{i}/{len(xbrl_files)}] 正在处理文件: {file_path}")
        result = process_xbrl_file(file_path)
        if result:
            results.append(result)
    
    # 创建结果DataFrame
    df_results = pd.DataFrame(results)
    
    # 保存结果
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_file = os.path.join(output_dir, f'analysis_results_{timestamp}_n{len(results)}.csv')
    df_results.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    return df_results

if __name__ == "__main__":
    results_df = process_all_files()
    print(f"处理完成，共分析 {len(results_df)} 个文件")
    print("\n结果预览：")
    print(results_df) 

2025-04-01 01:42:36,374 - INFO - 日志系统初始化完成
2025-04-01 01:42:36,383 - INFO - 将处理所有 505 个XBRL文件
2025-04-01 01:42:36,384 - INFO - [1/505] 正在处理文件: /Users/wenjun/Downloads/JPXData/AGM/XBRL/74220_20250318.xbrl
2025-04-01 01:42:36,389 - INFO - 找到公司名称: 東邦レマック株式会社
2025-04-01 01:42:36,392 - INFO - 处理文件: 74220_20250318 (公司: 東邦レマック株式会社)
2025-04-01 01:42:36,396 - INFO - 找到 1 个决议元素
2025-04-01 01:42:36,402 - INFO - 在决议元素 1 中找到 2 个表格
2025-04-01 01:42:36,403 - INFO - 原始列名: ['決議事項', '', '賛成数(個)', '反対数(個)', '棄権数(個)', '可決要件', '決議の結果及び賛成(反対)割合(％)', 'Column_8']
2025-04-01 01:42:36,405 - INFO - 处理表格 1 的列: ['決議事項', '可決要件', '決議の結果', '賛成割合（％）']
2025-04-01 01:42:36,405 - INFO - 找到赞成率: 99.95% (行 1)
2025-04-01 01:42:36,405 - INFO - 找到赞成率: 99.95% (行 2)
2025-04-01 01:42:36,406 - INFO - 找到赞成率: 99.87% (行 3)
2025-04-01 01:42:36,406 - INFO - 找到赞成率: 99.95% (行 4)
2025-04-01 01:42:36,406 - INFO - 找到赞成率: 99.92% (行 5)
2025-04-01 01:42:36,407 - INFO - 找到赞成率: 99.87% (行 6)
2025-04-01 01:42:36,407 - INFO - 找到赞成率: 99.85% (行 7)
20

处理完成，共分析 502 个文件

结果预览：
          file_name       company_name min_approval_rate rejection_flag
0    74220_20250318         東邦レマック株式会社             99.85              N
1    48830_20250328           株式会社モダリス             88.46              N
2    48960_20250328         株式会社ケイファーマ             99.86              N
3    43940_20250303       株式会社エクスモーション             99.56              N
4    36230_20250326       ビリングシステム株式会社             96.41              N
..              ...                ...               ...            ...
497  49710_20250326            メック株式会社             94.19              N
498  36490_20250328       株式会社ファインデックス             85.26              N
499  43740_20250327  株式会社ＲＯＢＯＴ　ＰＡＹＭＥＮＴ             99.77              N
500  47460_20250328           株式会社東計電算             92.32              N
501  40190_20250328           株式会社スタメン             99.64              N

[502 rows x 4 columns]


In [6]:
df2 = pd.read_csv('/Users/wenjun/Downloads/Quants/EDINET/data_2025-03-31.csv')
df2['secCode'] = df2['secCode'].astype(str).apply(lambda x: x.split('.')[0] if '.' in x else x)


# 确保数据类型正确
df2['docTypeCode'] = df2['docTypeCode'].astype(str)
df2['ordinanceCode'] = df2['ordinanceCode'].astype(str)
df2['formCode'] = df2['formCode'].astype(str)

# 修改后的筛选条件
filtered_df2 = df2[
    (df2['docTypeCode'].str.strip() == '180.0') & 
    (df2['ordinanceCode'].str.strip() == '10.0') & 
    (df2['formCode'].str.strip() == '053000') &
    (df2['secCode'].notna()) &
    (df2['currentReportReason'] == '第19条第2項第9号の2') &
    (df2['submitDateTime'] >= '2025-01-01')
]

/var/folders/bb/04y0ymcd2jz23qgx3f1q9n7c0000gn/T/ipykernel_4196/1423714968.py:1: DtypeWarning: Columns (3,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('/Users/wenjun/Downloads/Quants/EDINET/data_2025-03-31.csv')


In [24]:
df = results_df
df[['ticker', 'submission_date']] = df['file_name'].str.split('_', expand=True)
df

,file_name,min_approval_rate,rejection_flag,ticker,submission_date
0,74220_20250318,99.85,N,74220,20250318
1,48830_20250328,88.46,N,48830,20250328
2,48960_20250328,99.86,N,48960,20250328
3,43940_20250303,99.56,N,43940,20250303
4,36230_20250326,96.41,N,36230,20250326
...,...,...,...,...,...
497,49710_20250326,94.19,N,49710,20250326
498,36490_20250328,85.26,N,36490,20250328
499,43740_20250327,99.77,N,43740,20250327
500,47460_20250328,92.32,N,47460,20250328


In [25]:
filtered_df2

,seqNumber,docID,edinetCode,secCode,JCN,filerName,fundCode,ordinanceCode,formCode,docTypeCode,...,docInfoEditStatus,disclosureStatus,xbrlFlag,pdfFlag,attachDocFlag,englishDocFlag,csvFlag,legalStatus,date,submission_date
452140,95,S100V1DZ,E05181,94460,4.180001e+12,株式会社サカイホールディングス,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-01-06,20250106
452169,124,S100V12A,E27043,60870,8.011001e+12,株式会社アビスト,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-01-06,20250106
452171,126,S100V1FM,E34748,70630,8.010001e+12,株式会社Ｂｉｒｄｍａｎ,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-01-06,20250106
452361,73,S100V1Q3,E30648,60940,2.010401e+12,株式会社フリークアウト・ホールディングス,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-01-07,20250107
452801,127,S100V27X,E02762,99410,5.010001e+12,太洋物産株式会社,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-01-09,20250109
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468906,947,S100VIQM,E36708,40740,4.010401e+12,株式会社ラキール,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-03-28,20250328
468909,950,S100VIV9,E37160,42580,8.010001e+12,株式会社網屋,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-03-28,20250328
468916,957,S100VITB,E37466,50270,7.010401e+12,ＡｎｙＭｉｎｄ Ｇｒｏｕｐ株式会社,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-03-28,20250328
468927,968,S100VIFT,E02404,79560,8.010001e+12,ピジョン株式会社,NaN,10,053000,180,...,0,0,1,1,0,0,1,1,2025-03-28,20250328


In [26]:
# 确保filtered_df2中的secCode是字符串类型
filtered_df2['secCode'] = filtered_df2['secCode'].astype(str)

# 确保df中的ticker是字符串类型
df['ticker'] = df['ticker'].astype(str)

# 修改合并代码
result_df = pd.merge(
    df,
    filtered_df2[['secCode', 'filerName', 'docID', 'submission_date']],
    left_on=['ticker', 'submission_date'],
    right_on=['secCode', 'submission_date'],
    how='left'
)

# 整理最终列
final_df = result_df[[
    'ticker', 
    'filerName', 
    'min_approval_rate', 
    'rejection_flag',
    'submission_date',
    'docID'
]]

/var/folders/bb/04y0ymcd2jz23qgx3f1q9n7c0000gn/T/ipykernel_4196/691993704.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df2['secCode'] = filtered_df2['secCode'].astype(str)


In [27]:
final_df

,ticker,filerName,min_approval_rate,rejection_flag,submission_date,docID
0,74220,東邦レマック株式会社,99.85,N,20250318,S100VF1Y
1,48830,株式会社モダリス,88.46,N,20250328,S100VISH
2,48960,株式会社ケイファーマ,99.86,N,20250328,S100VIBE
3,43940,株式会社エクスモーション,99.56,N,20250303,S100VBWQ
4,36230,ビリングシステム株式会社,96.41,N,20250326,S100VH40
...,...,...,...,...,...,...
497,49710,メック株式会社,94.19,N,20250326,S100VGQ1
498,36490,株式会社ファインデックス,85.26,N,20250328,S100VI1W
499,43740,株式会社ＲＯＢＯＴ ＰＡＹＭＥＮＴ,99.77,N,20250327,S100VHKT
500,47460,株式会社東計電算,92.32,N,20250328,S100VIAC


In [28]:
import yfinance as yf
from tqdm import tqdm
# 获取市值数据
print("正在获取市值数据...")
final_df['market_cap'] = None  # 创建新列存储市值数据

def get_market_cap(code):
    try:
        # 将企业代码转换为yfinance格式（4位数字 + .T）
        # 首先确保是字符串格式，然后取前4位，补零，最后加上.T
        ticker = f"{str(code)[:4].zfill(4)}.T"
        stock = yf.Ticker(ticker)
        mcap = stock.info.get('marketCap', None)
        return mcap
    except:
        return None
# 使用tqdm添加进度条
for idx in tqdm(final_df.index):
    code = final_df.loc[idx, 'ticker']
    mcap = get_market_cap(code)
    final_df.loc[idx, 'market_cap'] = mcap

# 将市值转换为十亿日元单位
final_df['market_cap_billions'] = final_df['market_cap'] / 1_000_000_000

# 显示结果预览
print("\n数据预览：")
print(final_df[['ticker', 'market_cap_billions']].head())

# 显示数据获取成功率
total = len(final_df)
success = final_df['market_cap'].notna().sum()
print(f"\n数据获取成功率: {success}/{total} ({success/total*100:.1f}%)")
final_df = final_df.drop(columns=['market_cap'])
final_df['market_cap_billions'] = final_df['market_cap_billions'].round(1)

/var/folders/bb/04y0ymcd2jz23qgx3f1q9n7c0000gn/T/ipykernel_4196/1468847494.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['market_cap'] = None  # 创建新列存储市值数据


正在获取市值数据...


100%|██████████| 502/502 [04:20<00:00,  1.93it/s]


数据预览：
  ticker market_cap_billions
0  74220            2.208279
1  48830            5.968736
2  48960            9.457749
3  43940            2.593433
4  36230            7.761095

数据获取成功率: 490/502 (97.6%)


In [29]:
final_df.to_csv('interactive_table.csv', index=False)


In [43]:
# 读取数据
final_df = pd.read_csv("interactive_table.csv")

# 处理market_cap_billions列
final_df['market_cap_billions'] = pd.to_numeric(final_df['market_cap_billions'], errors='coerce')
final_df['market_cap_billions'] = final_df['market_cap_billions'].fillna(0)
final_df['market_cap_billions'] = final_df['market_cap_billions'].round(1)

# 处理min_approval_rate列
final_df['min_approval_rate'] = pd.to_numeric(final_df['min_approval_rate'], errors='coerce')
final_df['min_approval_rate'] = final_df['min_approval_rate'].fillna(0)
final_df['min_approval_rate'] = final_df['min_approval_rate'].round(2)  # 改为保留2位小数

# 创建docID的超链接
def create_edinet_link(doc_id):
    return f'<a href="https://api.edinet-fsa.go.jp/api/v2/documents/{doc_id}?type=2&Subscription-Key=6f25302ddba748ba866ae2164b963e02" target="_blank">{doc_id}</a>'

final_df['docID'] = final_df['docID'].apply(create_edinet_link)

# 生成HTML表格
html_string = '''
<html>
    <head>
        <meta charset="utf-8">
        <title>数据表格</title>
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.7/css/jquery.dataTables.css">
        <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/buttons/2.4.2/css/buttons.dataTables.min.css">
        <script type="text/javascript" src="https://code.jquery.com/jquery-3.7.0.js"></script>
        <script type="text/javascript" src="https://cdn.datatables.net/1.13.7/js/jquery.dataTables.min.js"></script>
        <script type="text/javascript" src="https://cdn.datatables.net/buttons/2.4.2/js/dataTables.buttons.min.js"></script>
        <style>
            body { 
                font-family: Arial, sans-serif; 
                margin: 20px;
                height: 100vh;
            }
            .dataTables_wrapper { 
                margin-top: 20px;
                height: calc(100vh - 100px);
            }
            .dataTables_scrollBody {
                max-height: calc(100vh - 200px) !important;
            }
            th { background-color: #f5f5f5; }
            td { padding: 8px; }
            a { color: #0066cc; text-decoration: none; }
            a:hover { text-decoration: underline; }
        </style>
    </head>
    <body>
        <h2>数据浏览器</h2>
        %s
        <script>
            $(document).ready(function() {
                $('#myTable').DataTable({
                    order: [],
                    scrollY: true,
                    scrollCollapse: true,
                    paging: true,
                    pageLength: 50,
                    lengthMenu: [[25, 50, 100, -1], [25, 50, 100, "全部"]],
                    columnDefs: [
                        {
                            targets: [2, 6],  // min_approval_rate和market_cap_billions列的索引
                            type: 'num',
                            render: function(data, type, row) {
                                if (type === 'sort') {
                                    return parseFloat(data.replace(/[^0-9.-]+/g, ''));
                                }
                                return data;
                            }
                        }
                    ],
                    language: {
                        "lengthMenu": "每页显示 _MENU_ 条记录",
                        "zeroRecords": "没有找到记录",
                        "info": "第 _PAGE_ 页 ( 总共 _PAGES_ 页 )",
                        "infoEmpty": "无记录",
                        "infoFiltered": "(从 _MAX_ 条记录过滤)",
                        "search": "搜索:",
                        "paginate": {
                            "first": "首页",
                            "last": "末页",
                            "next": "下一页",
                            "previous": "上一页"
                        }
                    }
                });
            });
        </script>
    </body>
</html>
'''

# 转换DataFrame为HTML表格
table_html = final_df.to_html(
    table_id='myTable', 
    index=False,
    classes=['display', 'compact'],
    border=0,
    escape=False,  # 允许HTML标签渲染
    float_format=lambda x: '{:.2f}'.format(x) if pd.notnull(x) else '0.00'  # 改为保留2位小数
)

# 保存为HTML文件
with open('interactive_table.html', 'w', encoding='utf-8') as f:
    f.write(html_string % table_html)